# Theta phase precession from button presses

MNE filters each participant's MEG epochs to 2–5 Hz and extracts analytic phase with Epochs.apply_hilbert. The circ_regress_db helper fits a circular-linear slope and returns the signed Kempter correlation for each complete trial and sensor. Trial values are averaged within participant, and the group scalp maps average participant means equally. Subject 11 is excluded before analysis.

UPPT002_button_press_times.npz stores cue-relative seconds in participant × trial × press-slot order. Subject 16 has 222 trials. Only trials with all five usable press times contribute, so press positions are comparable across trials.

The helper's per-trial permutation p-value is disabled during the sensor-by-trial fit because thousands of permutations for every sensor and trial would be very slow. Instead, the notebook uses MNE's one-sample permutation test across participant means, with false-discovery-rate correction across sensors. These group p-values test whether mean signed correlation or slope differs from zero; they are not the helper's per-trial phase-shuffle p-values.

import packages for MEG processing, circular regression and plotting

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from circ_regress_db import circ_regress_db

set data locations, theta parameters, subject exclusion and permutation count

In [ ]:
DATA_ROOT = os.environ.get(
    "MEG_DATA_ROOT",
    r"//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
)
BUTTON_PRESS_TIMES_FILE = os.path.join(
    os.getcwd(), "UPPT002_button_press_times.npz"
)

theta_freq_band = [2.0, 5.0]         # theta band edges in Hz
excluded_subjects = ["11"]           # participants excluded from analysis
CHANNEL_BATCH_SIZE = 8               # limit memory while filtering/Hilbert-transforming
N_GROUP_PERMUTATIONS = 5000           # group-level permutation tests

load button-press times and keep subject IDs aligned with their saved array rows

In [ ]:
def normalise_subject_id(id):
    return str(int(str(id).strip().lower().lstrip("s"))).zfill(2)


with np.load(BUTTON_PRESS_TIMES_FILE, allow_pickle=False) as press_file:
    UPPT002_times = np.asarray(press_file["uppt002_times"], dtype=float)
    subject_ID = np.asarray(
        [normalise_subject_id(id) for id in press_file["subject_ID"]]
    )
    n_trials_by_subject = np.asarray(
        press_file["n_trials_by_subject"], dtype=int
    )
    press_slot = np.asarray(press_file["press_slot"], dtype=int)
    CUE_ONSET_AFTER_EPOCH_START_S = float(
        np.asarray(press_file["cue_onset_after_epoch_start_s"]).item()
    )
    time_reference = str(press_file["time_reference"].item())

if "relative to cue onset" not in time_reference.lower():
    raise ValueError("Expected button-press times in seconds relative to cue onset")
if UPPT002_times.ndim != 3 or UPPT002_times.shape[0] != len(subject_ID):
    raise ValueError("Unexpected participant × trial × press-slot array shape")
if not np.array_equal(press_slot, np.arange(1, len(press_slot) + 1)):
    raise ValueError("Expected press slots numbered in sequence from 1")

n_press_slots = len(press_slot)
sub_ID = [id for id in subject_ID if id not in excluded_subjects]
subject_index = {id: index for index, id in enumerate(subject_ID)}
if not sub_ID:
    raise ValueError("No participants remain after applying excluded_subjects.")

print("Array shape:", UPPT002_times.shape)
print("Participants:", sub_ID)
print("Excluded participants:", excluded_subjects)

load MEG epochs using the repository filename pattern

In [ ]:
def filename_loader(id):
    subject_number = str(int(id))
    subject_folder = "s" + id
    file_name = "Subject" + subject_number + "_spm_mne-epo.fif"
    epochs_file = os.path.join(DATA_ROOT, subject_folder, file_name)

    if not os.path.isfile(epochs_file):
        epochs_file = os.path.join(DATA_ROOT, file_name)
    if not os.path.isfile(epochs_file):
        raise FileNotFoundError("Could not find " + file_name + " under " + DATA_ROOT)

    epochs = mne.read_epochs(epochs_file, preload=False, verbose="ERROR")
    MEG_sens = [channel for channel in epochs.ch_names if channel.startswith("M")]
    if not MEG_sens:
        raise ValueError("No MEG sensors found in " + epochs_file)

    return epochs, MEG_sens


def mean_finite(values, axis=0):
    values = np.asarray(values)
    counts = np.isfinite(values).sum(axis=axis)
    return np.divide(
        np.nansum(values, axis=axis),
        counts,
        out=np.full(counts.shape, np.nan, dtype=np.result_type(values.dtype, float)),
        where=counts > 0,
    )

filter MEG channels with MNE, extract phase at each press, and apply circ_regress_db per trial and sensor

In [ ]:
def participant_phase_precession(id):
    epochs, MEG_sens = filename_loader(id)
    data_index = subject_index[id]
    n_trials = int(n_trials_by_subject[data_index])

    if len(epochs) != n_trials:
        raise ValueError(
            "Participant " + id + " has " + str(len(epochs))
            + " MEG epochs but " + str(n_trials) + " timing rows"
        )

    press_times = UPPT002_times[data_index, :n_trials]
    cue_onset_epoch_s = epochs.times[0] + CUE_ONSET_AFTER_EPOCH_START_S
    event_times = press_times + cue_onset_epoch_s
    sample_indices = np.zeros(press_times.shape, dtype=int)
    valid_press = np.isfinite(event_times)
    sample_indices[valid_press] = np.rint(
        (event_times[valid_press] - epochs.times[0]) * epochs.info["sfreq"]
    ).astype(int)
    valid_press &= (
        (sample_indices >= 0) & (sample_indices < len(epochs.times))
    )
    complete_trial = valid_press.sum(axis=1) == n_press_slots

    trial_slope = np.full((n_trials, len(MEG_sens)), np.nan)
    trial_phase = np.full_like(trial_slope, np.nan)
    trial_corr = np.full_like(trial_slope, np.nan)

    for start in range(0, len(MEG_sens), CHANNEL_BATCH_SIZE):
        channel_batch = MEG_sens[start:start + CHANNEL_BATCH_SIZE]
        epochs_batch = epochs.copy().pick(channel_batch).load_data()
        epochs_batch.filter(
            l_freq=theta_freq_band[0],
            h_freq=theta_freq_band[1],
            picks=channel_batch,
            verbose="ERROR",
        ).apply_hilbert(
            picks=channel_batch,
            envelope=False,
            verbose="ERROR",
        )
        analytic = epochs_batch.get_data()

        for trial in np.flatnonzero(complete_trial):
            samples = sample_indices[trial]
            for channel_index in range(len(channel_batch)):
                phase = np.angle(analytic[trial, channel_index, samples])
                result = circ_regress_db(press_slot, phase, k=0, ppdir=0)
                trial_slope[trial, start + channel_index] = result["slope_opt"]
                trial_phase[trial, start + channel_index] = result["phase_opt"]
                trial_corr[trial, start + channel_index] = result["corr"]

        del analytic
        del epochs_batch

    # Average trial values within this participant; phase offsets use a circular mean.
    participant_slope = mean_finite(trial_slope, axis=0)
    participant_phase = np.angle(
        mean_finite(np.exp(1j * trial_phase), axis=0)
    )
    participant_corr = mean_finite(trial_corr, axis=0)

    picks = mne.pick_channels(epochs.ch_names, include=MEG_sens)
    sensor_info = mne.pick_info(epochs.info, picks, copy=True)
    summary = {
        "participant": id,
        "n_trials": n_trials,
        "n_complete_press_trials": int(complete_trial.sum()),
        "n_valid_press_times": int(valid_press.sum()),
        "n_trials_with_corr": int(np.isfinite(trial_corr).any(axis=1).sum()),
    }
    return (
        participant_slope, participant_corr, participant_phase,
        MEG_sens, sensor_info, summary
    )

process included participants; the helper's per-trial permutation p-values are disabled here

In [ ]:
participant_slope = []
participant_corr = []
participant_phase = []
participant_rows = []
group_sensors = None
group_info = None

for id in sub_ID:
    slope, corr, phase, MEG_sens, sensor_info, summary = (
        participant_phase_precession(id)
    )
    if group_sensors is None:
        group_sensors, group_info = MEG_sens, sensor_info
    elif MEG_sens != group_sensors:
        raise ValueError("MEG sensor order differs for participant " + id)

    participant_slope.append(slope)
    participant_corr.append(corr)
    participant_phase.append(phase)
    participant_rows.append(summary)
    print(
        "Participant", id,
        "- complete trials:", summary["n_complete_press_trials"],
    )

participant_slope = np.asarray(participant_slope)
participant_corr = np.asarray(participant_corr)
participant_phase = np.asarray(participant_phase)
participant_summary_table = pd.DataFrame(participant_rows)

group_slope = mean_finite(participant_slope, axis=0)
group_corr = mean_finite(participant_corr, axis=0)
group_phase = np.angle(
    mean_finite(np.exp(1j * participant_phase), axis=0)
)

test participant means against zero and correct p-values across sensors

In [ ]:
def group_permutation_test(values):
    # Test participant means against zero; correct across sensors with FDR.
    t_value = np.full(values.shape[1], np.nan)
    p_value = np.full(values.shape[1], np.nan)
    rng = np.random.RandomState(0)

    for sensor_index in range(values.shape[1]):
        sensor_values = values[:, sensor_index]
        sensor_values = sensor_values[np.isfinite(sensor_values)]
        if len(sensor_values) < 3:
            continue

        t_value[sensor_index], p_value[sensor_index], _ = (
            mne.stats.permutation_t_test(
                sensor_values[:, None],
                n_permutations=N_GROUP_PERMUTATIONS,
                tail=0,
                seed=rng,
                verbose="ERROR",
            )
        )

    p_fdr = np.full_like(p_value, np.nan)
    valid = np.isfinite(p_value)
    if valid.any():
        _, p_fdr[valid] = mne.stats.fdr_correction(p_value[valid], alpha=0.05)

    return t_value, p_value, p_fdr


corr_t, corr_p, corr_p_fdr = group_permutation_test(participant_corr)
slope_t, slope_p, slope_p_fdr = group_permutation_test(participant_slope)

participant_corr_table = pd.DataFrame(
    participant_corr, index=sub_ID, columns=group_sensors
)
participant_slope_table = pd.DataFrame(
    participant_slope, index=sub_ID, columns=group_sensors
)
participant_phase_table = pd.DataFrame(
    participant_phase, index=sub_ID, columns=group_sensors
)

group_results = pd.DataFrame({
    "sensor": group_sensors,
    "mean_corr": group_corr,
    "corr_t": corr_t,
    "corr_p": corr_p,
    "corr_p_fdr": corr_p_fdr,
    "mean_slope_rad_per_press": group_slope,
    "slope_t": slope_t,
    "slope_p": slope_p,
    "slope_p_fdr": slope_p_fdr,
    "mean_phase_offset_rad": group_phase,
})

display(participant_summary_table.round(4))
display(participant_corr_table.round(3))
display(participant_slope_table.round(3))
display(group_results.round(4))

plot group mean correlation and slope for each MEG sensor type

In [ ]:
sensor_types = [
    mne.channel_type(group_info, index)
    for index in range(len(group_sensors))
]

for values, label, limits in [
    (group_corr, "Mean signed circular correlation", (-1.0, 1.0)),
    (group_slope, "Mean slope (radians per press)", (-2 * np.pi, 2 * np.pi)),
]:
    for sensor_type in sorted(set(sensor_types)):
        picks = np.array([
            index for index, kind in enumerate(sensor_types)
            if kind == sensor_type
        ])
        sensor_info = mne.pick_info(group_info, picks, copy=True)
        fig, ax = plt.subplots(figsize=(7, 6))
        image, _ = mne.viz.plot_topomap(
            values[picks],
            sensor_info,
            ch_type=sensor_type,
            axes=ax,
            cmap="RdBu_r",
            contours=0,
            show=False,
        )
        image.set_clim(*limits)
        ax.set_title(label + " (" + sensor_type + ")")
        fig.colorbar(image, ax=ax)
        plt.tight_layout()
        plt.show()